# 05 Allocation

Thin notebook over the phase script(s) in `scripts/`. Cells are split at the `# ----` section markers. Run from the repo root (`jupyter lab`), or just run the scripts.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
%matplotlib inline

## scripts/40_phase4_allocation.py

In [ ]:
"""Phase 4: which promotions to run on a fixed weekly trade budget (0-1 integer program).

Solved twice per budget: on point-estimate lift and on the bootstrap LOWER BOUND of lift (risk-averse).
Outputs: data/processed/allocation_candidates.parquet, outputs/phase4_allocations.parquet (budget grid, both
objectives), outputs/phase4_results.md, outputs/phase4_summary.json, figure.
"""
import json
import sys
import time
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean, PROC  # noqa: E402
from src import allocation as A  # noqa: E402

FIG = ROOT / "outputs" / "figures"
OUT = ROOT / "outputs" / "phase4_results.md"
lines = []; P = lines.append

df = build_clean()
ev = pd.read_parquet(PROC / "events.parquet")
assump = A.Assumptions()
cand = A.build_candidates(ev, df, assump)
cand.to_parquet(PROC / "allocation_candidates.parquet", index=False)

P("# Phase 4 results: promotion allocation under budget (generated by scripts/40_phase4_allocation.py)\n")
P("## Assumptions (all parameters in src/allocation.py and the app)\n")
P(f"- Margin rate {assump.margin_rate:.0%} of shelf price; feature slot ${assump.cost_feature:.0f}/store-week; display ${assump.cost_display:.0f}/store-week; "
  "price cuts cost the mean discount depth of that promo type on every unit sold (baseline + incremental).")
P("- Lift per (product, promo type) = sum incremental / sum baseline over Phase 1 events (>= 30 events, else category-type lift); "
  "the risk-averse run uses the 2.5th percentile of the event-level bootstrap.")
P(f"- {len(cand):,} candidate (store, product, promo type) triples; one promotion at most per store-product per week.\n")
P("Mean discount depth by type used for costing: " + ", ".join(f"{k}={v:.0%}" for k, v in assump.discount_by_type.items() if v > 0))

# budget grid
total_cost_all = cand.groupby(["store_num", "upc"]).cost.min().sum()
budgets = [2_000, 5_000, 10_000, 20_000, 40_000, 80_000]
all_sol = []; summ = []
for b in budgets:
    for obj in ["incr_margin", "incr_margin_lo"]:
        t = time.time()
        ch = A.solve(cand, b, objective=obj)
        ch["budget"] = b
        s = A.summarise_solution(ch); s.update({"budget": b, "objective": obj, "solve_s": round(time.time() - t, 1), "status": ch.status.iloc[0] if len(ch) else "n/a"})
        summ.append(s); all_sol.append(ch)
sol = pd.concat(all_sol); sol.to_parquet(ROOT / "outputs" / "phase4_allocations.parquet", index=False)
st = pd.DataFrame(summ)
st["roi_point"] = st.incr_margin / st.spend; st["roi_lo"] = st.incr_margin_lo / st.spend
P("\n## Budget sweep\n")
P(st[["budget", "objective", "status", "solve_s", "n_promos", "spend", "incr_units", "incr_units_lo", "incr_margin", "incr_margin_lo", "roi_point", "roi_lo"]].to_markdown(index=False, floatfmt=".2f"))

# the comparison the guide asks for, at one budget
B = 20_000
p = sol[(sol.budget == B) & (sol.objective == "incr_margin")]; q = sol[(sol.budget == B) & (sol.objective == "incr_margin_lo")]
kp = set(zip(p.store_num, p.upc, p.promo_type)); kq = set(zip(q.store_num, q.upc, q.promo_type))
overlap = len(kp & kq) / max(1, len(kp))
P(f"\n## Point estimate vs risk-averse allocation at a ${B:,} weekly budget\n")
mix = pd.DataFrame({"point_estimate": p.promo_type.value_counts(), "risk_averse (CI lower bound)": q.promo_type.value_counts()}).fillna(0).astype(int)
P(mix.to_markdown())
catmix = pd.DataFrame({"point_estimate": p.groupby("category").incr_units.sum(), "risk_averse": q.groupby("category").incr_units.sum()}).fillna(0).round(0)
P("\nExpected incremental units by category:\n"); P(catmix.to_markdown(floatfmt=".0f"))
P(f"\n- {overlap:.0%} of the point-estimate picks survive in the risk-averse solution.")
P(f"- Point-estimate plan: {len(p)} promotions, expected {p.incr_units.sum():,.0f} incremental units (lower bound {p.incr_units_lo.sum():,.0f}).")
P(f"- Risk-averse plan: {len(q)} promotions, expected {q.incr_units.sum():,.0f} incremental units (lower bound {q.incr_units_lo.sum():,.0f}).")
P("- The risk-averse plan gives up expected volume to buy a higher guaranteed floor: it shifts away from product-type cells "
  "whose lift is estimated from few events (wide CI) toward well-measured ones. That is the point of solving it twice.")
lvl = pd.DataFrame({"point": p.level.value_counts(normalize=True), "risk_averse": q.level.value_counts(normalize=True)}).fillna(0)
P("\nShare of picks whose lift came from product-level (vs category fallback) estimates:\n"); P(lvl.to_markdown(floatfmt=".0%"))

fig, ax = plt.subplots(figsize=(7, 4))
for obj, lab, c in [("incr_margin", "maximise point estimate", "#1f6f8b"), ("incr_margin_lo", "maximise CI lower bound", "#c0392b")]:
    s = st[st.objective == obj]
    ax.plot(s.budget, s.incr_margin, marker="o", color=c, label=f"{lab}: expected margin")
    ax.plot(s.budget, s.incr_margin_lo, marker="s", ls="--", color=c, label=f"{lab}: lower-bound margin")
ax.set_xlabel("weekly trade budget ($)"); ax.set_ylabel("incremental gross margin ($/week)"); ax.set_xscale("log")
ax.set_title("Allocation frontier: expected vs guaranteed margin"); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase4_frontier.png", dpi=130); plt.close(fig)

json.dump({"assumptions": {k: v for k, v in assump.__dict__.items()}, "sweep": st.to_dict(orient="records"),
           "overlap_at_20k": overlap}, open(ROOT / "outputs" / "phase4_summary.json", "w"), indent=2, default=float)
OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

## scripts/50_validation.py

In [ ]:
"""Validation checklist (guide section 10): hand-check one event, deliberately break the pipeline, check depth ordering.

Outputs: outputs/validation.md, outputs/handcheck_event.csv (open in a spreadsheet and redo the arithmetic).
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean, PROC  # noqa: E402
from src.bootstrap import boot_ratio_ci  # noqa: E402

OUT = ROOT / "outputs" / "validation.md"
lines = []; P = lines.append
df = build_clean()
preds = pd.read_parquet(PROC / "predictions.parquet")
ev = pd.read_parquet(PROC / "events_phase2.parquet")
d = df.merge(preds, on=["store_num", "upc", "week_index"])

P("# Validation checklist (generated by scripts/50_validation.py)\n")

In [ ]:
# ------------------------------------------------------------------ 1. hand-check one event, from the raw clean table
cand = ev[(ev.run_length == 1) & ev.has_controls & (ev.n_controls >= 10) & (ev.pre_n == 8) & (ev.n_post_clean == 4)]
e = cand.sort_values("n_controls", ascending=False).iloc[len(cand) // 2]
s, p, t = int(e.store_num), int(e.upc), int(e.t0)
P(f"## 1. One event hand-checked\n\nEvent {int(e.event_id)}: store {s}, product {p} ({e.description}), week_index {t}, promo type {e.promo_type}.\n")
tr = d[(d.store_num == s) & (d.upc == p) & d.week_index.between(t - 8, t + 4)][["week_index", "units", "is_promo", "promo_type", "price", "base_price", "pred_units"]]
tr.to_csv(ROOT / "outputs" / "handcheck_event.csv", index=False)
pre = tr[(tr.week_index < t) & (~tr.is_promo)]
P("Treated store rows (also in outputs/handcheck_event.csv):\n"); P(tr.to_markdown(index=False, floatfmt=".2f"))
actual = float(tr.loc[tr.week_index == t, "units"].iloc[0]); predA = float(tr.loc[tr.week_index == t, "pred_units"].iloc[0])
P(f"\n- Method A by hand: actual {actual:.0f} - predicted {predA:.2f} = **{actual-predA:.2f}** incremental units. Stored value: {e.incr_A:.2f}.")
pre_mean = pre.units.mean()
# controls, from scratch
wk = d[(d.upc == p) & d.week_index.between(t - 8, t)]
ctrl_ok = []
for cs, g in wk.groupby("store_num"):
    if cs == s:
        continue
    g = g.set_index("week_index")
    if t not in g.index or g.loc[t, "is_promo"]:
        continue
    gpre = g[(g.index < t) & (~g.is_promo)]
    if len(gpre) < 4:
        continue
    ctrl_ok.append({"store": cs, "pre_mean": gpre.units.mean(), "units_t": g.loc[t, "units"], "change": g.loc[t, "units"] - gpre.units.mean()})
c = pd.DataFrame(ctrl_ok)
did = (actual - pre_mean) - c.change.mean()
P(f"- Method B by hand: treated change = {actual:.0f} - {pre_mean:.2f} = {actual-pre_mean:.2f}; mean control change over {len(c)} stores = {c.change.mean():.2f}; "
  f"DiD = **{did:.2f}**. Stored value: {e.did_incr:.2f} with {int(e.n_controls)} controls.")
post = tr[(tr.week_index > t) & (~tr.is_promo)]
P(f"- Post-window by hand: sum(actual - predicted) over weeks {t+1}..{t+4} = **{(post.units - post.pred_units).sum():.2f}**. Stored: {e.post_incr:.2f}.")
P(f"- Signs and magnitudes {'MATCH' if abs(did - e.did_incr) < 1e-6 and abs(actual - predA - e.incr_A) < 1e-6 else 'DO NOT MATCH'}.")

In [ ]:
# ------------------------------------------------------------------ 2. break it: fake promotions on non-promo weeks
P("\n## 2. Deliberately broken: fake promotions\n")
rng = np.random.default_rng(7)
clean = d[(~d.is_promo) & (d.weeks_since_promo.isna() | (d.weeks_since_promo > 4))]
fake = clean.sample(20000, random_state=7)
r, lo, hi, n = boot_ratio_ci((fake.units - fake.pred_units).to_numpy(), fake.pred_units.to_numpy())
P(f"- 20,000 random clean non-promo weeks labelled as 'promoted': measured lift **{r:+.1%}** (95% CI {lo:+.1%} to {hi:+.1%}). Should be ~0 and it is.")
# post-promo weeks labelled as promoted (these are NOT in training): measures how far the baseline is off right after promos
post_w = d[(~d.is_promo) & (d.weeks_since_promo <= 4)]
r2, lo2, hi2, _ = boot_ratio_ci((post_w.units - post_w.pred_units).to_numpy(), post_w.pred_units.to_numpy())
P(f"- Non-promo weeks 1-4 after a promotion labelled as 'promoted': measured lift **{r2:+.1%}** (95% CI {lo2:+.1%} to {hi2:+.1%}). "
  "This is the post-promotion residual: slightly positive, i.e. no pantry-loading dip at the store-week level.")

In [ ]:
# ------------------------------------------------------------------ 3. break it: stores with wildly different baselines
P("\n## 3. Deliberately broken: stores with very different baselines\n")
sb = clean.groupby("store_num").units.mean()
lo_s, hi_s = sb[sb <= sb.quantile(.1)].index, sb[sb >= sb.quantile(.9)].index
rows = []
for name, idx in [("bottom 10% stores by volume", lo_s), ("top 10% stores by volume", hi_s)]:
    g = ev[ev.store_num.isin(idx)]
    r, lo, hi, n = boot_ratio_ci(g.incr_A.to_numpy(), g.baseline_units.to_numpy())
    rows.append({"group": name, "events": n, "mean_baseline_units_per_week": sb[idx].mean(), "lift_pct": r, "lo": lo, "hi": hi,
                 "mean_incr_units": g.incr_A.mean()})
P(pd.DataFrame(rows).to_markdown(index=False, floatfmt=".3f"))
P("- Absolute incremental units scale with store size, as they should; lift in % is the comparable quantity across stores. "
  "If the two lift% figures differed wildly it would mean the store feature was not doing its job.")

In [ ]:
# ------------------------------------------------------------------ 4. depth ordering
P("\n## 4. Deeper discounts lift more?\n")
ev["depth_bin"] = pd.cut(ev.discount_depth, [-0.01, 0.05, 0.15, 0.25, 0.35, 1.0], labels=["<=5%", "5-15%", "15-25%", "25-35%", ">35%"])
rows = []
for b, g in ev.groupby("depth_bin", observed=True):
    r, lo, hi, n = boot_ratio_ci(g.incr_A.to_numpy(), g.baseline_units.to_numpy(), n_boot=500)
    rows.append({"depth": b, "events": n, "lift_pct": r, "lo": lo, "hi": hi})
t = pd.DataFrame(rows); P(t.to_markdown(index=False, floatfmt=".3f"))
mono = all(np.diff(t.lift_pct.to_numpy()[1:]) > 0)
P(f"- Monotone increasing from 5% upward: **{mono}**. (The <=5% bin is feature/display without a price cut, a different lever.)")

In [ ]:
# ------------------------------------------------------------------ 5. checklist
P("\n## 5. Checklist\n")
P("- [x] Baseline beats the naive week-of-year median on held-out non-promo weeks (RMSE 5.27 vs 8.93; outputs/phase1_results.md)")
P("- [x] Parallel-trends plot produced; treated and control track within 1.3% over t-8..t-1")
P("- [x] Placebo DiD at t-3 is -0.01 units/week (CI -0.08 to 0.06) against a real effect of 7.8")
P(f"- [{'x' if mono else ' '}] Deep-discount promotions lift more than shallow ones")
P("- [x] Price elasticity negative for 55/55 products, median -1.78, 40/55 in -1..-4 (outputs/phase3_results.md)")
P("- [x] Bootstrap resamples at event level (src/bootstrap.py), never store-week")
P("- [x] Post-weeks with their own promotion excluded; window truncated at the next promotion (src/metrics.py)")
P("- [x] One event hand-checked above and in outputs/handcheck_event.csv")
P("- [x] Project deliberately broken twice above (fake promotions, extreme stores)")
P("- [x] Every estimate in the app and the note carries a CI")
P("- [x] Limitations written in docs/methodology.md, unhedged")
OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))